# Module 2 · Lecture 2 — Segmentation with a U-Net

Hands-on companion to the *Lecture 2* slides. We train a **U-Net** to segment
polyps in endoscopy images, following the same universal recipe as Lecture 1 —
only the **model** (U-Net), **loss** (BCE + Dice), and **output** (a per-pixel mask)
change.

**Dataset:** [Kvasir-SEG](https://datasets.simula.no/kvasir-seg/) — ~1000 RGB polyp
images with binary masks (~46 MB). Downloads directly; runs on a **free Colab GPU**.

> Set `Runtime → Change runtime type → GPU`. `QUICK_RUN = True` trains 1 epoch on a
> subset (~1 min smoke test); set `False` for a real run.

## 0 · Setup

In [ ]:
import os, random, urllib.request, zipfile
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms.functional as TF
from PIL import Image
import matplotlib.pyplot as plt

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
QUICK_RUN = True   # 1 epoch on a subset for a fast end-to-end check

## 1 · Data — Kvasir-SEG

Download + unzip once. Images and masks share filenames; masks are white-on-black,
which we binarize. Everything is resized to 128×128.

In [ ]:
URL = 'https://datasets.simula.no/downloads/kvasir-seg.zip'
if not os.path.isdir('Kvasir-SEG'):
    print('downloading Kvasir-SEG (~46 MB) ...')
    urllib.request.urlretrieve(URL, 'kvasir-seg.zip')
    with zipfile.ZipFile('kvasir-seg.zip') as z:
        z.extractall('.')
img_dir, mask_dir = 'Kvasir-SEG/images', 'Kvasir-SEG/masks'
files = sorted(os.listdir(img_dir))
print('images:', len(files))

In [ ]:
SIZE = 128

class KvasirSeg(Dataset):
    def __init__(self, names, size=SIZE, augment=False):
        self.names, self.size, self.augment = names, size, augment
    def __len__(self):
        return len(self.names)
    def __getitem__(self, i):
        name = self.names[i]
        img = Image.open(os.path.join(img_dir, name)).convert('RGB').resize((self.size, self.size))
        msk = Image.open(os.path.join(mask_dir, name)).convert('L').resize((self.size, self.size), Image.NEAREST)
        if self.augment and random.random() < 0.5:
            img, msk = img.transpose(Image.FLIP_LEFT_RIGHT), msk.transpose(Image.FLIP_LEFT_RIGHT)
        x = TF.to_tensor(img)                       # (3,H,W) in [0,1]
        m = (TF.to_tensor(msk) > 0.5).float()       # (1,H,W) binary
        return x, m

random.shuffle(files)
n_val = int(0.15 * len(files))
val_names, train_names = files[:n_val], files[n_val:]
train_ds = KvasirSeg(train_names, augment=True)
val_ds   = KvasirSeg(val_names)
print('train/val:', len(train_ds), len(val_ds))

In [ ]:
BATCH = 16
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False)
xb, mb = next(iter(train_loader))
print('batch:', xb.shape, mb.shape, 'mask fg fraction:', round(mb.mean().item(), 3))

## 2 · Model — U-Net

Encoder (downsample, *what*) + decoder (upsample, *where*), with **skip
connections that concatenate** encoder features into the decoder.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, cin=3, cout=1, feats=(32, 64, 128)):
        super().__init__()
        self.downs = nn.ModuleList(); self.ups = nn.ModuleList()
        self.pool = nn.MaxPool2d(2)
        prev = cin
        for f in feats:
            self.downs.append(DoubleConv(prev, f)); prev = f
        self.bottleneck = DoubleConv(feats[-1], feats[-1]*2)
        prev = feats[-1]*2
        for f in reversed(feats):
            self.ups.append(nn.ConvTranspose2d(prev, f, 2, stride=2))
            self.ups.append(DoubleConv(prev, f))   # prev = f(up) + f(skip)
            prev = f
        self.head = nn.Conv2d(feats[0], cout, 1)
    def forward(self, x):
        skips = []
        for down in self.downs:
            x = down(x); skips.append(x); x = self.pool(x)
        x = self.bottleneck(x)
        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            skip = skips[-(i//2 + 1)]
            x = torch.cat([skip, x], dim=1)        # <-- concat skip connection
            x = self.ups[i+1](x)
        return self.head(x)

unet = UNet(cin=3, cout=1).to(device)
print(sum(p.numel() for p in unet.parameters())/1e6, 'M params')
print('output:', unet(torch.randn(2, 3, SIZE, SIZE, device=device)).shape)

## 3 · Loss & metric — Dice (+ BCE)

Dice is robust when the foreground (polyp) is small relative to background.

In [ ]:
def dice_loss(logits, target, eps=1.0):
    prob = torch.sigmoid(logits)
    num = 2 * (prob * target).sum(dim=(2, 3)) + eps
    den = prob.sum(dim=(2, 3)) + target.sum(dim=(2, 3)) + eps
    return (1 - num/den).mean()

def combined_loss(logits, target):
    return F.binary_cross_entropy_with_logits(logits, target) + dice_loss(logits, target)

@torch.no_grad()
def dice_score(logits, target, eps=1e-6):
    pred = (torch.sigmoid(logits) > 0.5).float()
    num = 2 * (pred * target).sum(dim=(1, 2, 3))
    den = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    return ((num + eps) / (den + eps)).mean()

## 4 · Train

In [ ]:
opt = torch.optim.Adam(unet.parameters(), lr=1e-3)

def run_epoch(loader, train: bool):
    unet.train(train)
    loss_sum, dice_sum, n = 0.0, 0.0, 0
    with torch.set_grad_enabled(train):
        for x, m in loader:
            x, m = x.to(device), m.to(device)
            logits = unet(x)
            loss = combined_loss(logits, m)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            bs = x.size(0)
            loss_sum += loss.item() * bs
            dice_sum += dice_score(logits, m).item() * bs
            n += bs
    return loss_sum/n, dice_sum/n

In [ ]:
EPOCHS = 1 if QUICK_RUN else 15
if QUICK_RUN:
    fit_loader = DataLoader(Subset(train_ds, list(range(100))), batch_size=BATCH, shuffle=True)
else:
    fit_loader = train_loader

for epoch in range(EPOCHS):
    tr_loss, tr_dice = run_epoch(fit_loader, train=True)
    va_loss, va_dice = run_epoch(val_loader, train=False)
    print(f'epoch {epoch+1}/{EPOCHS}  train_loss={tr_loss:.3f} dice={tr_dice:.3f}  |  '
          f'val_loss={va_loss:.3f} val_dice={va_dice:.3f}')

## 5 · Evaluate & visualize

Overlay *image | ground truth | prediction* for a few validation cases and save the
montage to `../Figures/kvasir_overlay.png` — the Lecture-2 *Hands-on* slide picks it
up automatically.

In [ ]:
unet.eval()
x, m = next(iter(val_loader))
with torch.no_grad():
    pred = (torch.sigmoid(unet(x.to(device))) > 0.5).float().cpu()

rows = min(4, x.size(0))
fig, axes = plt.subplots(rows, 3, figsize=(5, 1.6*rows))
for r in range(rows):
    axes[r, 0].imshow(x[r].permute(1, 2, 0).numpy())
    axes[r, 1].imshow(m[r, 0].numpy(), cmap='gray')
    axes[r, 2].imshow(pred[r, 0].numpy(), cmap='gray')
for ax in axes.ravel(): ax.axis('off')
for c, t in enumerate(['image', 'ground truth', 'prediction']):
    axes[0, c].set_title(t, fontsize=8)
plt.tight_layout()

fig_dir = os.path.join('..', 'Figures'); os.makedirs(fig_dir, exist_ok=True)
out_png = os.path.join(fig_dir, 'kvasir_overlay.png')
plt.savefig(out_png, dpi=150, bbox_inches='tight', facecolor='white')
print('saved', out_png)
plt.show()

## Recap & next steps
- Same recipe as Lecture 1, with a U-Net, BCE + Dice loss, and a per-pixel output.
- **Try:** `QUICK_RUN = False` for a full run (Dice ≈ 0.8–0.9), stronger augmentation,
  or a deeper `feats=(32,64,128,256)`.
- **Swap dataset:** point `img_dir`/`mask_dir` at ISIC lesions or lung-CXR masks —
  the model and loss are unchanged.

Next: **synthesis** — generate new images with a VAE and a GAN.